# 04 · Prediction models, outreach, and subgroup audit

Fits the three models for **no recorded BP measurement within 90 days**, evaluates them on the test set, runs nested cross-validation, the outreach-capacity table, decision curves, the subgroup (equity) audit, group-specific thresholds, alternative targets, calibration, and Shapley values.

**Reads (from notebook 03, in `DATA_DIR`):** `pred_cohort`, `pred_features`

**Produces (patient-level, `DATA_DIR`):** `pred_split_90d`, `pred_test_90d`, `pred_oof_90d`

**Produces (aggregate, `results/`):** CSVs for Table 2A–B, Table 3 (point estimates), Tables S5–S6, decision curve, calibration deciles, detection by group, Shapley importance; `04_prediction_models.json`

**Paper items:** Table 2; Figure 2A–B; Figure S1; Tables S5–S6; Shapley results in the text. (Table 3 confidence intervals and the final figures are drawn in notebook 08.)

Runtime: about 30–45 minutes on a standard Colab CPU (the tuning searches and nested cross-validation are the slow parts).

## 0 · Setup

Patients are sorted by ID before splitting and cross-validation, and every random step uses seed 42, so results are repeatable.

In [ ]:
# ---- edit these ----
REPO_DIR = "/content/drive/MyDrive/Ronveer Sandhu/Project 1 -MIMIC analysis-hypertension retention/htn-bp-followup-mimic"
DATA_DIR = "/content/drive/MyDrive/htn_bp_data"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install xgboost==3.4.1 shap==0.52.* statsmodels==0.15.0 scikit-learn==1.6.1

import os, sys, json, itertools, time
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from htn_utils import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from xgboost import XGBClassifier

RESULTS_DIR = os.path.join(REPO_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

P = load_df(DATA_DIR, "pred_cohort")
F = load_df(DATA_DIR, "pred_features")

## 1 · Features and the 90-day sample

The 62 features, in the order used to fit the models:

In [ ]:
FEATS = ['age_index', 'sbp_baseline', 'dbp_baseline', 'creatinine', 'egfr', 'bmi', 'hba1c', 'hba1c_missing',
         'bmi_missing', 'n_ed_prior', 'n_inp_prior', 'n_enc_prior', 'n_antihtn_drugs', 'elixhauser_vw',
         'aids', 'alcohol', 'blane', 'carit', 'chf', 'coag', 'cpd', 'dane', 'depre', 'diabc', 'diabunc', 'drug',
         'fed', 'hypc', 'hypothy', 'hypunc', 'ld', 'lymph', 'metacanc', 'obes', 'ond', 'para', 'pcd', 'psycho',
         'pud', 'pvd', 'rf', 'rheumd', 'solidtum', 'valv', 'wloss',
         'n_prior_enc', 'recency_days', 'interval_entropy', 'mean_interval', 'sbp_mean', 'sbp_sd', 'sbp_trend',
         'gender_F', 'gender_M', 'race_Asian', 'race_Black', 'race_Hispanic', 'race_Other/Unknown', 'race_White',
         'insurance_Medicaid', 'insurance_Medicare', 'insurance_Other']
assert len(FEATS) == 62

Fi = F.set_index("subject_id")
X_all = pd.concat([Fi.drop(columns=["gender", "race", "insurance"]),
                   pd.get_dummies(Fi[["gender", "race", "insurance"]], dtype=float)], axis=1)
missing = [f for f in FEATS if f not in X_all.columns]
assert not missing, f"features not built: {missing}"
X_all = X_all[FEATS].astype(float)

P = P.sort_values("subject_id").reset_index(drop=True)
Pi = P.set_index("subject_id")

y90 = Pi.target_90.dropna().astype(int)
print(f"90-day sample: {len(y90):,} | no recorded BP within 90 days: {int(y90.sum()):,} ({100 * y90.mean():.1f}%)")

### Train / validation / test split (60 / 20 / 20, stratified by outcome)

Set assignments were made first in the 270-day sample, so both targets share test patients; the 1,815 patients eligible only for the 90-day target were then assigned at random in the same proportions (Supplementary Methods).

In [ ]:
def split3(ids, yv):
    rest, te = train_test_split(ids, test_size=0.2, random_state=42, stratify=yv.loc[ids])
    tr, va   = train_test_split(rest, test_size=0.25, random_state=42, stratify=yv.loc[rest])
    return pd.concat([pd.Series("train", index=tr), pd.Series("val", index=va), pd.Series("test", index=te)])

y270 = Pi.target_270.dropna().astype(int)
set270 = split3(y270.index, y270)
new_ids = y90.index.difference(y270.index)
set90 = pd.concat([set270, split3(new_ids, y90)]).loc[y90.index]

print(pd.crosstab(set90, y90, margins=True, colnames=["no BP in 90 days"]))

## 2 · Fit the three models (Table 2A)

- **Two-variable logistic**: days since last visit and prior encounters (effectively unpenalized)
- **Penalized logistic regression**: L2 penalty chosen from 10 values by five-fold CV on AUPRC
- **Gradient-boosted trees (XGBoost)**: 16 combinations of learning rate, depth, minimum child weight, and number of trees, chosen by five-fold CV on AUPRC

Logistic models use median imputation and standardization fitted within the training data; XGBoost handles missing values directly. Each model is recalibrated with isotonic regression on the validation set and evaluated once on the test set; confidence intervals come from 1,000 bootstrap samples of the test set.

In [ ]:
X = X_all.loc[y90.index]; y = y90
itr, iva, ite = (set90 == "train").values, (set90 == "val").values, (set90 == "test").values
cv = StratifiedKFold(5, shuffle=True, random_state=42)
t0 = time.time()

def lr_pipe(C):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=C, max_iter=5000))

TWO = ["recency_days", "n_prior_enc"]
m_two = lr_pipe(1e6).fit(X.loc[itr, TWO], y[itr])

Cs = np.logspace(-3, 3, 10)
lr_scores = {C: cross_val_score(lr_pipe(C), X[itr], y[itr], cv=cv, scoring="average_precision").mean() for C in Cs}
C_best = max(lr_scores, key=lr_scores.get)
m_lr = lr_pipe(C_best).fit(X[itr], y[itr])
print(f"penalized logistic: C = {C_best:.4f} (CV AUPRC {lr_scores[C_best]:.3f})  [{time.time()-t0:.0f}s]")

grid = list(itertools.product([0.03, 0.1], [3, 5], [1, 5], [200, 400]))
xgb_scores = {}
for lr_, d, mcw, n in grid:
    m = XGBClassifier(learning_rate=lr_, max_depth=d, min_child_weight=mcw, n_estimators=n,
                      tree_method="hist", eval_metric="aucpr", random_state=42, n_jobs=-1)
    xgb_scores[(lr_, d, mcw, n)] = cross_val_score(m, X[itr], y[itr], cv=cv, scoring="average_precision").mean()
best = max(xgb_scores, key=xgb_scores.get)
xgb_best = dict(zip(["learning_rate", "max_depth", "min_child_weight", "n_estimators"], best))
m_xgb = XGBClassifier(**xgb_best, tree_method="hist", eval_metric="aucpr",
                      random_state=42, n_jobs=-1).fit(X[itr], y[itr])
print(f"XGBoost: {xgb_best} (CV AUPRC {xgb_scores[best]:.3f})  [{time.time()-t0:.0f}s]")

In [ ]:
MODELS = ["XGBoost", "Penalized logistic", "Two-variable"]
fitted = {"Two-variable": (m_two, TWO), "Penalized logistic": (m_lr, FEATS), "XGBoost": (m_xgb, FEATS)}

test90 = pd.DataFrame({"subject_id": y.index[ite], "y": y[ite].values})
for name, (m, cols) in fitted.items():
    pv = m.predict_proba(X.loc[iva, cols])[:, 1]
    pt = m.predict_proba(X.loc[ite, cols])[:, 1]
    iso = IsotonicRegression(out_of_bounds="clip").fit(pv, y[iva])
    test90[f"raw_{name}"] = pt
    test90[f"cal_{name}"] = iso.predict(pt)

rng = np.random.default_rng(42)
B  = [rng.integers(0, len(test90), len(test90)) for _ in range(1000)]
yt = test90.y.values

def cal_int_slope(p, yv):
    lp = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    slope = sm.GLM(yv, sm.add_constant(lp), family=sm.families.Binomial()).fit().params[1]
    inter = sm.GLM(yv, np.ones_like(lp), family=sm.families.Binomial(), offset=lp).fit().params[0]
    return inter, slope

perf = {}
rows = []
for name in MODELS:
    r, p = test90[f"raw_{name}"].values, test90[f"cal_{name}"].values
    au = [roc_auc_score(yt[b], r[b]) for b in B]
    ap = [average_precision_score(yt[b], r[b]) for b in B]
    ci, cs = cal_int_slope(p, yt)
    perf[name] = dict(auroc=roc_auc_score(yt, r), auroc_lo=np.percentile(au, 2.5), auroc_hi=np.percentile(au, 97.5),
                      auprc=average_precision_score(yt, r), auprc_lo=np.percentile(ap, 2.5), auprc_hi=np.percentile(ap, 97.5),
                      brier=brier_score_loss(yt, p), cal_intercept=ci, cal_slope=cs)
    rows.append({"model": name, **{k: round(v, 3) for k, v in perf[name].items()}})

rx = test90["raw_XGBoost"].values
for comp in ["Penalized logistic", "Two-variable"]:
    rc = test90[f"raw_{comp}"].values
    dd = [roc_auc_score(yt[b], rx[b]) - roc_auc_score(yt[b], rc[b]) for b in B]
    perf[comp].update(diff=roc_auc_score(yt, rx) - roc_auc_score(yt, rc),
                      diff_lo=np.percentile(dd, 2.5), diff_hi=np.percentile(dd, 97.5))
brier_prev = brier_score_loss(yt, np.full(len(yt), y[itr].mean()))

table2a = pd.DataFrame([{"model": m, **{k: round(v, 3) for k, v in perf[m].items()}} for m in MODELS])
print(f"test set n = {len(yt):,}; no recorded BP = {yt.sum():,} ({100*yt.mean():.1f}%)")
print(f"prevalence-only Brier: {brier_prev:.3f}")
print(table2a.to_string(index=False))

## 3 · Outreach at fixed capacity (Table 2B) and decision curve (Figure 2A)

At each capacity level, patients are ranked by raw predicted risk and the top 5%, 10%, 20%, or 30% are flagged. The decision curve uses the recalibrated predictions.

In [ ]:
CAPS = [0.05, 0.10, 0.20, 0.30]

def capacity_stats(score, yv, cap):
    k = int(round(cap * len(yv)))
    flag = np.zeros(len(yv), bool); flag[np.argsort(-score, kind="stable")[:k]] = True
    tp = (flag & (yv == 1)).sum()
    return {"found_pct": 100 * tp / yv.sum(), "ppv_pct": 100 * tp / k,
            "contacts_per_found": k / tp if tp else np.nan, "found_n": tp, "flagged_n": k}

rows = []
for m in MODELS:
    s = test90[f"raw_{m}"].values
    for cap in CAPS:
        est = capacity_stats(s, yt, cap)
        bs = pd.DataFrame([capacity_stats(s[b], yt[b], cap) for b in B])
        row = {"model": m, "top_pct": int(cap * 100), "flagged": est["flagged_n"], "found": est["found_n"]}
        for col in ["found_pct", "ppv_pct", "contacts_per_found"]:
            row[col] = est[col]; row[f"{col}_lo"] = bs[col].quantile(.025); row[f"{col}_hi"] = bs[col].quantile(.975)
        rows.append(row)
table2b = pd.DataFrame(rows)
print(table2b.round(1).to_string(index=False))
print(f"\nrandom selection: PPV {100*yt.mean():.1f}%, {1/yt.mean():.1f} contacts per patient found")

In [ ]:
TH = np.round(np.arange(0.05, 0.71, 0.01), 2)
def net_benefit(p, yv, t):
    f = p >= t
    return ((f & (yv == 1)).sum() - (f & (yv == 0)).sum() * t / (1 - t)) / len(yv)
def nb_all(yv, t):
    return yv.mean() - (1 - yv.mean()) * t / (1 - t)

curves, bands = {}, {}
for m in MODELS:
    p = test90[f"cal_{m}"].values
    curves[m] = np.array([net_benefit(p, yt, t) for t in TH])
    boot = np.array([[net_benefit(p[b], yt[b], t) for t in TH] for b in B])
    bands[m] = np.percentile(boot, [2.5, 97.5], axis=0)
treat_all = np.array([nb_all(yt, t) for t in TH])

dca = pd.DataFrame({"threshold": TH, "treat_all": treat_all,
                    **{f"nb_{m}": curves[m] for m in MODELS},
                    **{f"lo_{m}": bands[m][0] for m in MODELS},
                    **{f"hi_{m}": bands[m][1] for m in MODELS}})

dca_range, dca_range_band = {}, {}
for m in MODELS:
    better = TH[curves[m] > np.maximum(treat_all, 0)]
    better_b = TH[bands[m][0] > np.maximum(treat_all, 0)]
    dca_range[m] = (float(better.min()), float(better.max())) if len(better) else None
    dca_range_band[m] = (float(better_b.min()), float(better_b.max())) if len(better_b) else None
    print(f"{m}: net benefit above both defaults at {dca_range[m]}; lower 95% band above both at {dca_range_band[m]}")

fig, ax = plt.subplots(figsize=(7, 4.8))
for m in MODELS:
    ax.fill_between(TH, *bands[m], alpha=0.18, lw=0)
    ax.plot(TH, curves[m], lw=2, label=m)
ax.plot(TH, treat_all, color="0.55", ls="--", label="Contact everyone")
ax.axhline(0, color="0.2", lw=1, label="Contact no one")
ax.set_xlim(0.05, 0.60); ax.set_ylim(-0.03, 0.34)
ax.set_xlabel("Risk threshold for outreach"); ax.set_ylabel("Net benefit")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

## 4 · Nested cross-validation (out-of-fold predictions)

Five outer folds, each with a three-fold tuning search (12 combinations). Each fold's predictions are recalibrated with an isotonic fit to the other four folds, so no patient's own outcome calibrates their score. These out-of-fold predictions are used for the subgroup audit.

In [ ]:
t0 = time.time()
yv = y90.values
outer = StratifiedKFold(5, shuffle=True, random_state=42)
inner = StratifiedKFold(3, shuffle=True, random_state=42)
grid9 = list(itertools.product([0.03], [4, 5, 6], [1, 5], [200, 400]))

oof_raw = np.zeros(len(yv)); fold_id = np.zeros(len(yv), int); fold_best = []
for f, (tr, te) in enumerate(outer.split(X, yv)):
    sc = {}
    for lr_, d, mcw, n in grid9:
        m = XGBClassifier(learning_rate=lr_, max_depth=d, min_child_weight=mcw, n_estimators=n,
                          tree_method="hist", eval_metric="aucpr", random_state=42, n_jobs=-1)
        sc[(lr_, d, mcw, n)] = cross_val_score(m, X.iloc[tr], yv[tr], cv=inner, scoring="average_precision").mean()
    b = max(sc, key=sc.get)
    hp = dict(zip(["learning_rate", "max_depth", "min_child_weight", "n_estimators"], b))
    fold_best.append(hp)
    m = XGBClassifier(**hp, tree_method="hist", eval_metric="aucpr", random_state=42, n_jobs=-1).fit(X.iloc[tr], yv[tr])
    oof_raw[te] = m.predict_proba(X.iloc[te])[:, 1]; fold_id[te] = f
    print(f"fold {f}: {hp}  inner AUPRC {sc[b]:.3f}  [{time.time()-t0:.0f}s]")

oof_cal = np.zeros(len(yv))
for f in range(5):
    o = fold_id != f
    oof_cal[~o] = IsotonicRegression(out_of_bounds="clip").fit(oof_raw[o], yv[o]).predict(oof_raw[~o])

def boot_auc(yy, p, n=500, seed=42):
    r = np.random.default_rng(seed); a_, p_ = [], []
    for _ in range(n):
        b = r.integers(0, len(yy), len(yy))
        a_.append(roc_auc_score(yy[b], p[b])); p_.append(average_precision_score(yy[b], p[b]))
    return np.percentile(a_, [2.5, 97.5]), np.percentile(p_, [2.5, 97.5])

(oa_lo, oa_hi), (op_lo, op_hi) = boot_auc(yv, oof_raw)
oof_ci, oof_cs = cal_int_slope(oof_cal, yv)
nested = dict(auroc=roc_auc_score(yv, oof_raw), auroc_lo=oa_lo, auroc_hi=oa_hi,
              auprc=average_precision_score(yv, oof_raw), auprc_lo=op_lo, auprc_hi=op_hi,
              cal_intercept=oof_ci, cal_slope=oof_cs)
print({k: round(v, 3) for k, v in nested.items()})

## 5 · Subgroup audit at 20% overall selection (Table 3, Figure 2B)

The 20% of patients with the highest out-of-fold risk are flagged. By group: AUROC, observed/predicted ratio and calibration slope (recalibrated predictions), share flagged, true and false positive rates, PPV, and the difference in true positive rate from the reference group (500 bootstrap samples). Confidence intervals for every Table 3 measure are added in notebook 08.

In [ ]:
G = Fi.loc[y90.index]
grp = pd.DataFrame({
    "Race": G.race.values,
    "Insurance": G.insurance.values,
    "Sex": G.gender.map({"F": "Female", "M": "Male"}).values,
    "Age": pd.cut(G.age_index, [0, 49, 64, 79, 200], labels=["<50", "50–64", "65–79", "≥80"]).astype(str).values})
REF = {"Race": "White", "Insurance": "Other", "Sex": "Female", "Age": "65–79"}

thr = np.quantile(oof_raw, 0.80)
flag = oof_raw >= thr
target_tpr = flag[yv == 1].mean()
print(f"top-20% threshold (raw score) {thr:.3f}; flagged {100*flag.mean():.1f}%; overall detection {target_tpr:.3f}")

def gstats(idx):
    yy, ff, pr_, pc = yv[idx], flag[idx], oof_raw[idx], oof_cal[idx]
    return {"n": int(idx.sum()), "outcome_rate": yy.mean(),
            "auroc": roc_auc_score(yy, pr_) if 0 < yy.sum() < len(yy) else np.nan,
            "obs_pred": yy.mean() / pc.mean(), "cal_slope": cal_int_slope(pc, yy)[1],
            "flagged": ff.mean(), "tpr": ff[yy == 1].mean(), "fpr": ff[yy == 0].mean(),
            "ppv": yy[ff].mean() if ff.any() else np.nan}

rng = np.random.default_rng(42)
BB = [rng.integers(0, len(yv), len(yv)) for _ in range(500)]
rows = []
for dim, ref in REF.items():
    g = grp[dim].values
    for lev in sorted(pd.unique(g)):
        st = gstats(g == lev)
        if lev == ref:
            st.update(tpr_diff=np.nan, tpr_diff_lo=np.nan, tpr_diff_hi=np.nan)
        else:
            diffs = [fb[(gb == lev) & (yb == 1)].mean() - fb[(gb == ref) & (yb == 1)].mean()
                     for yb, fb, gb in ((yv[b], flag[b], g[b]) for b in BB)]
            st.update(tpr_diff=st["tpr"] - flag[(g == ref) & (yv == 1)].mean(),
                      tpr_diff_lo=np.nanpercentile(diffs, 2.5), tpr_diff_hi=np.nanpercentile(diffs, 97.5))
        rows.append({"group": dim, "level": lev, "reference": lev == ref, **st})
table3 = pd.DataFrame(rows)
print(table3.round(3).to_string(index=False))

In [ ]:
# Table S6: single threshold vs race-specific thresholds that match overall detection
g_race = grp["Race"].values
rows, flag_grp = [], flag.copy()
for lev in sorted(pd.unique(g_race)):
    i = g_race == lev
    pos = np.sort(oof_raw[i & (yv == 1)])[::-1]
    t_g = pos[int(np.ceil(target_tpr * len(pos))) - 1]
    fg = oof_raw[i] >= t_g
    flag_grp[i] = fg
    rows.append({"race": lev, "n": int(i.sum()), "no_bp": int(yv[i].sum()),
                 "single_thr": thr, "group_thr": t_g,
                 "single_tpr": flag[i & (yv == 1)].mean(), "group_tpr": fg[yv[i] == 1].mean(),
                 "single_fpr": flag[i & (yv == 0)].mean(), "group_fpr": fg[yv[i] == 0].mean(),
                 "single_ppv": yv[i][flag[i]].mean(), "group_ppv": yv[i][fg].mean()})
tableS6 = pd.DataFrame(rows)
print(tableS6.round(3).to_string(index=False))
print(f"\ntotal flagged: single threshold {100*flag.mean():.1f}% -> race-specific thresholds {100*flag_grp.mean():.1f}%")

# Figure 2B data: detection by group with bootstrap CIs
order = [("Race", ["White", "Black", "Hispanic", "Asian", "Other/Unknown"]),
         ("Insurance", ["Other", "Medicare", "Medicaid"]),
         ("Sex", ["Female", "Male"]),
         ("Age", ["<50", "50–64", "65–79", "≥80"])]
plot_rows = []
for dim, levs in order:
    g = grp[dim].values
    for lev in levs:
        i = (g == lev) & (yv == 1)
        bs = [flag[b][(g[b] == lev) & (yv[b] == 1)].mean() for b in BB]
        plot_rows.append((dim, lev, flag[i].mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5), int(i.sum())))
fig2b = pd.DataFrame(plot_rows, columns=["dim", "level", "tpr", "lo", "hi", "no_bp_n"])
print(fig2b.round(3).to_string(index=False))

## 6 · Alternative targets and death as a competing event (Table S5)

**A.** Out-of-fold AUROC and AUPRC for each alternative target, using the tuning settings chosen for the 90-day target (no nested search). The last column applies the 90-day out-of-fold scores to each target's patients.

**B.** Cumulative incidence (Aalen–Johansen) of a BP reading and of death before any reading within 90 days, by fifth of predicted risk, among all 31,390 patients with data after the prediction date (out-of-fold scores, plus the full-data model for the 448 patients outside the 90-day sample); censored at the end of coverage.

In [ ]:
HP = xgb_best
def oof_fit(Xs, ys):
    p = np.zeros(len(ys))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=42).split(Xs, ys):
        m = XGBClassifier(**HP, tree_method="hist", eval_metric="aucpr", random_state=42, n_jobs=-1)
        p[te] = m.fit(Xs.iloc[tr], ys[tr]).predict_proba(Xs.iloc[te])[:, 1]
    return p

alt = {"No BP within 30 days": "target_30", "No BP within 90 days": "target_90",
       "No BP within 180 days": "target_180", "No BP within 270 days": "target_270",
       "No BP within 365 days": "target_365", "No BP within 90 days, deaths before BP counted": "target_90_deaths"}
prim = pd.Series(oof_raw, index=y90.index)
rows = []
for name, col in alt.items():
    s = Pi[col].dropna().astype(int)
    ys = s.values
    p = oof_fit(X_all.loc[s.index], ys)
    (alo, ahi), (plo, phi) = boot_auc(ys, p)
    common = s.index.intersection(prim.index)
    rows.append({"target": name, "n": len(ys), "outcomes": int(ys.sum()),
                 "auroc": roc_auc_score(ys, p), "auroc_lo": alo, "auroc_hi": ahi,
                 "auprc": average_precision_score(ys, p), "auprc_lo": plo, "auprc_hi": phi,
                 "auroc_90d_scores": roc_auc_score(s[common], prim[common]) if col != "target_90" else np.nan})
    print(f"  {name}: done")
tableS5a = pd.DataFrame(rows)
print(tableS5a.round(3).to_string(index=False))

In [ ]:
m_full = XGBClassifier(**HP, tree_method="hist", eval_metric="aucpr", random_state=42,
                       n_jobs=-1).fit(X_all.loc[y90.index], y90.values)
others = Pi.index[~Pi.index.isin(y90.index)]
score_all = pd.concat([prim, pd.Series(m_full.predict_proba(X_all.loc[others])[:, 1], index=others)])
print(f"panel B n = {len(score_all):,} ({len(prim):,} out-of-fold + {len(others):,} scored by the full model)")

ids = score_all.index
tb = (Pi.next_bp_day - Pi.idx_day).loc[ids]; tb = tb.where(tb > 0)
td = (Pi.days_to_death - Pi.idx_day).loc[ids]
ta = (Pi.admin_day - Pi.idx_day).loc[ids]
time_ = pd.concat([tb, td, ta], axis=1).min(axis=1)
event = np.select([tb.notna() & (tb <= time_), td.notna() & (td <= time_)], [1, 2], 0)
fifth = pd.qcut(score_all.rank(method="first"), 5, labels=[1, 2, 3, 4, 5])

def aj_cif(t, e, H=90):
    d = pd.DataFrame({"t": t, "e": e}).sort_values("t")
    S, c1, c2, n = 1.0, 0.0, 0.0, len(d)
    for tt, g_ in d.groupby("t"):
        if tt > H: break
        d1, d2 = (g_.e == 1).sum(), (g_.e == 2).sum()
        c1 += S * d1 / n; c2 += S * d2 / n
        S *= 1 - (d1 + d2) / n; n -= len(g_)
    return c1, c2

rows = []
for q in range(1, 6):
    i = (fifth == q).values
    c1, c2 = aj_cif(time_.values[i], event[i])
    rows.append({"fifth": q, "n": int(i.sum()), "bp_pct": 100 * c1, "death_before_bp_pct": 100 * c2,
                 "neither_pct": 100 * (1 - c1 - c2)})
tableS5b = pd.DataFrame(rows)
print(tableS5b.round(1).to_string(index=False))

## 7 · Calibration (Figure S1) and Shapley values

In [ ]:
cal_rows = []
for m in MODELS:
    p = test90[f"cal_{m}"].values
    dec = pd.qcut(pd.Series(p).rank(method="first"), 10, labels=False)
    d = pd.DataFrame({"p": p, "y": yt, "dec": dec}).groupby("dec").agg(pred=("p", "mean"), obs=("y", "mean"))
    for _, r_ in d.iterrows():
        cal_rows.append({"model": m, "pred": r_.pred, "obs": r_.obs})
figS1 = pd.DataFrame(cal_rows)

import shap
Xte = X.loc[ite]
sv = shap.TreeExplainer(m_xgb).shap_values(Xte)
shap_imp = (pd.DataFrame({"feature": Xte.columns, "mean_abs_shap": np.abs(sv).mean(0)})
              .sort_values("mean_abs_shap", ascending=False).reset_index(drop=True))
shap_imp["share_pct"] = 100 * shap_imp.mean_abs_shap / shap_imp.mean_abs_shap.sum()
shap_imp["direction"] = [pd.Series(Xte[f].values).corr(pd.Series(sv[:, list(Xte.columns).index(f)]), method="spearman")
                         for f in shap_imp.feature]
print(shap_imp.head(10).round(3).to_string(index=False))

## 8 · Results summary

The numbers reported in the paper, in one place.

In [ ]:
f3 = lambda d, k: f"{d[k]:.3f} ({d[k + '_lo']:.3f}-{d[k + '_hi']:.3f})"
print(f"TEST SET: n = {len(yt):,}; no recorded BP = {yt.sum():,} ({100 * yt.mean():.1f}%); prevalence-only Brier {brier_prev:.3f}")
for m in MODELS:
    d = perf[m]
    line = (f"  {m:20s} AUROC {f3(d, 'auroc')} | AUPRC {f3(d, 'auprc')} | Brier {d['brier']:.3f} | "
            f"calibration {d['cal_intercept']:.2f} / {d['cal_slope']:.2f}")
    if "diff" in d:
        line += f" | AUROC difference vs XGBoost {d['diff']:.3f} ({d['diff_lo']:.3f}-{d['diff_hi']:.3f})"
    print(line)
print(f"  Nested cross-validation (all {len(yv):,}): AUROC {f3(nested, 'auroc')} | AUPRC {f3(nested, 'auprc')} | "
      f"calibration {nested['cal_intercept']:.2f} / {nested['cal_slope']:.2f}")

print("\nOUTREACH AT FIXED CAPACITY (XGBoost; found % / PPV % / contacts per patient found)")
for _, r_ in table2b[table2b.model == "XGBoost"].iterrows():
    other = table2b[(table2b.top_pct == r_.top_pct) & (table2b.model != "XGBoost")].set_index("model").found_pct
    print(f"  top {r_.top_pct:>2}%: flagged {r_.flagged:,}, found {r_.found:,} | "
          f"{r_.found_pct:.1f} ({r_.found_pct_lo:.1f}-{r_.found_pct_hi:.1f}) | "
          f"{r_.ppv_pct:.1f} ({r_.ppv_pct_lo:.1f}-{r_.ppv_pct_hi:.1f}) | "
          f"{r_.contacts_per_found:.1f} ({r_.contacts_per_found_lo:.1f}-{r_.contacts_per_found_hi:.1f}) | "
          f"found %, penalized / two-variable: {other['Penalized logistic']:.1f} / {other['Two-variable']:.1f}")
print(f"  decision curve, XGBoost above both defaults: estimate {dca_range['XGBoost']}, "
      f"lower 95% band {dca_range_band['XGBoost']}")

print(f"\nSUBGROUPS (top 20% flagged; threshold {thr:.3f}; overall detection {target_tpr:.3f})")
print(table3[["group", "level", "n", "outcome_rate", "auroc", "obs_pred", "cal_slope", "flagged", "tpr", "fpr", "ppv",
              "tpr_diff", "tpr_diff_lo", "tpr_diff_hi"]].round(3).to_string(index=False))
print(f"  race-specific thresholds: total flagged {100 * flag.mean():.1f}% -> {100 * flag_grp.mean():.1f}%")

print("\nALTERNATIVE TARGETS")
print(tableS5a[["target", "n", "outcomes", "auroc", "auroc_lo", "auroc_hi", "auprc", "auroc_90d_scores"]].round(3).to_string(index=False))
print(tableS5b.round(1).to_string(index=False))
print(f"\nSHAPLEY: top features {shap_imp.feature.head(4).tolist()}; "
      f"days since last visit {shap_imp.set_index('feature').loc['recency_days', 'share_pct']:.0f}% of total")

## 9 · Save

In [ ]:
# patient-level (never committed)
save_df(set90.rename("set").to_frame().assign(y90=y90).reset_index(), DATA_DIR, "pred_split_90d")
save_df(test90, DATA_DIR, "pred_test_90d")
save_df(pd.DataFrame({"subject_id": y90.index, "y": yv, "fold": fold_id, "oof": oof_raw, "oof_cal": oof_cal}),
        DATA_DIR, "pred_oof_90d")

# aggregate results
for df_, name in [(table2a, "table2a_discrimination"), (table2b, "table2b_capacity"), (dca, "figure2a_decision_curve"),
                  (table3, "table3_subgroups_point"), (tableS6, "tableS6_group_thresholds"), (fig2b, "figure2b_detection"),
                  (tableS5a, "tableS5a_alternative_targets"), (tableS5b, "tableS5b_competing_death"),
                  (figS1, "figureS1_calibration"), (shap_imp, "shapley_importance")]:
    df_.to_csv(os.path.join(RESULTS_DIR, f"{name}.csv"), index=False)
print("saved result tables to", RESULTS_DIR)

save_json({"lr_C": float(C_best), "xgb_best": xgb_best, "nested_fold_best": fold_best,
           "nested": {k: float(v) for k, v in nested.items()},
           "prevalence_brier": float(brier_prev),
           "top20_threshold_raw": float(thr), "overall_detection": float(target_tpr),
           "flagged_single": float(flag.mean()), "flagged_race_specific": float(flag_grp.mean()),
           "dca_xgb_range": dca_range["XGBoost"], "dca_xgb_range_lower_band": dca_range_band["XGBoost"]},
          RESULTS_DIR, "04_prediction_models")